# Адаптивные 150 кандидатов + CatBoost
Набор генераторов и веса RRF зависят от query_type. Политика зафиксирована по validation в `recall150_artifacts/adaptive_policy.json`: Recall@150 = 70,85% на validation и 68,33% на public sample. CatBoost обучаем поверх **всех оставшихся кандидатов**, максимум 150 на запрос; без семплирования негативов и принудительного добавления цели.

Сохраняем прежние 30 000 обучающих, 4 000 validation и 10 000 public-test запросов. Разбиение текстов и безопасные индексы неизменны. В итоговой оценке учитываются запросы с отсутствующей целью. Public test уже использовался для отчёта о фиксированной политике кандидатов; параметры этого ранкера по public test не подбираются.

58 признаков: прежние 32, признаки всех 11 источников и их общего RRF, плюс оценка и ранг выбранной адаптивной политики. Чтобы получить признаки, вычисляем все 11 источников; в выдачу допускаются только треки выбранного для типа запроса набора.

## Результат обучения

CatBoost обучен на 30 000 запросов, с адаптивным отбором до 150 кандидатов и 58 фичами. Validation nDCG@20 = **0,419214**, recall кандидатов = **0,7085**. На public test (10 000 запросов): nDCG@20 = **0,413624**, Hit@20 = **0,5575**, recall кандидатов = **0,6833**. Модель и параметры сохранены в `adaptive150_catboost_artifacts/`; действующие индексы — в `retrieval_artifacts/`.

## 1. Настройки

In [1]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["OMP_NUM_THREADS"] = "6"
import gc, re, json, hashlib
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import bm25s, joblib
from scipy.sparse import load_npz, save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from IPython.display import display

SEED = 42
VALID_LIMIT = 4000
TEST_LIMIT = 10000
BATCH_SIZE = 64
MAX_K = 1000
BUDGETS = [100, 200, 500, 1000]
# Работает при запуске из корня проекта или папки датасета.
PROJECT_DIR = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "music4all_crs" / "train_queries.parquet").exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Не найдена папка music4all_crs с исходными данными")
DATA_DIR = PROJECT_DIR / "music4all_crs"
BASE_DIR = PROJECT_DIR / "retrieval_artifacts"
SEARCH_DIR = BASE_DIR
CACHE_DIR = SEARCH_DIR / "candidate_cache"
OUT_DIR = PROJECT_DIR / "adaptive150_catboost_artifacts"
OUT_DIR.mkdir(exist_ok=True)

from catboost import CatBoostRanker, Pool
from sklearn.feature_extraction.text import CountVectorizer

TRAIN_LIMIT = 30000
ITERATIONS = 350
THREADS = 6
CANDIDATE_LIMIT = 150
POLICY_DIR = OUT_DIR
CAP_CONFIG = json.loads((POLICY_DIR / "candidate_policy.json").read_text())
TOP_K = 1000
RRF_CONSTANT = 60
SOURCE_NAMES = json.loads((SEARCH_DIR / "generators.json").read_text())[
    "generators"
]
OLD_FEATURES = json.loads((BASE_DIR / "base_feature_names.json").read_text())
FEATURES = (
    OLD_FEATURES
    + [f"{name}_source_rank" for name in SOURCE_NAMES]
    + [f"{name}_source_hit" for name in SOURCE_NAMES]
    + ["all_source_count", "all_rrf_score", "selection_rrf_score", "selection_rank"]
)
CAT_FEATURES = ["query_type"]
NUMERIC_FEATURES = [f for f in FEATURES if f not in CAT_FEATURES]
print("Features:", len(FEATURES), "generators:", len(SOURCE_NAMES))


Features: 58 generators: 11


## 2. Разбиение запросов

In [2]:
def normalize(text):
    return " ".join(re.findall(r"\w+", str(text).casefold()))


def text_bucket(text):
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16) % 10


def sample_queries(frame, limit):
    if limit is None or len(frame) <= limit:
        return frame.reset_index(drop=True)
    # Случайная выборка сохраняет естественное распределение типов.
    return frame.sample(n=limit, random_state=SEED).reset_index(drop=True)


train_inputs = pd.read_parquet(DATA_DIR / "train_queries.parquet")
test_inputs = pd.read_parquet(DATA_DIR / "test_public_queries.parquet")
assert train_inputs.query_id.is_unique and test_inputs.query_id.is_unique
train_inputs["normalized"] = train_inputs["query"].map(normalize)
test_texts = set(test_inputs["query"].map(normalize))
buckets = train_inputs.normalized.map(text_bucket)
index_inputs = train_inputs[
    (buckets < 6) & ~train_inputs.normalized.isin(test_texts)
].copy()
rank_inputs = sample_queries(train_inputs[(buckets >= 6) & (buckets < 9)], TRAIN_LIMIT)
valid_inputs = sample_queries(train_inputs[buckets == 9], VALID_LIMIT)
test_inputs = sample_queries(test_inputs, TEST_LIMIT)
assert not set(index_inputs.normalized) & set(rank_inputs.normalized)
assert not set(index_inputs.normalized) & set(valid_inputs.normalized)
assert not set(rank_inputs.normalized) & set(valid_inputs.normalized)
print(
    {
        "index_queries": len(index_inputs),
        "ranker_train": len(rank_inputs),
        "validation": len(valid_inputs),
        "public_test": len(test_inputs),
    }
)
display(test_inputs.query_type.value_counts().rename("test_queries").to_frame())


{'index_queries': 259505, 'ranker_train': 30000, 'validation': 4000, 'public_test': 10000}


,test_queries
query_type,
similar_to,1087
situation,804
era_region,796
complex,795
genre,792
audio_attributes,791
mood,782
vague_recall,771
exact,757


## 3. Каталог и безопасные документы

In [3]:
META_COLUMNS = [
    "m4a_id",
    "m4a_artist",
    "m4a_song",
    "m4a_album",
    "m4a_genres_full",
    "m4a_tags_full",
    "pseudo_caption",
    "lyrics",
    "artist_country",
    "lang",
    "release_year",
    "spotify_popularity",
    "danceability",
    "energy",
    "valence",
    "tempo",
    "duration_ms",
]
tracks = pd.read_parquet(DATA_DIR / "tracks_meta.parquet", columns=META_COLUMNS)
tracks = tracks.sort_values("m4a_id").reset_index(drop=True)
assert tracks.m4a_id.is_unique
track_ids = tracks.m4a_id.to_numpy(dtype=str)
id_to_idx = {track_id: i for i, track_id in enumerate(track_ids)}
train_qrels = pd.read_parquet(DATA_DIR / "train_qrels.parquet")
index_rows = index_inputs.merge(
    train_qrels[["query_id", "target_m4a_id"]], on="query_id", validate="one_to_one"
)
query_docs = (
    index_rows.drop_duplicates(["target_m4a_id", "normalized"])
    .groupby("target_m4a_id")["query"]
    .agg(" ".join)
)
tracks["query_document"] = tracks.m4a_id.map(query_docs).fillna("")
TEXT_COLUMNS = [
    "m4a_artist",
    "m4a_song",
    "m4a_album",
    "m4a_genres_full",
    "m4a_tags_full",
    "pseudo_caption",
    "lyrics",
    "artist_country",
    "lang",
    "release_year",
    "query_document",
]
documents = tracks[TEXT_COLUMNS].fillna("").astype(str).agg(" ".join, axis=1).tolist()
print(
    f"Треков: {len(tracks):,}; с train-запросами: {(tracks.query_document != '').sum():,}"
)
del index_inputs, train_inputs, query_docs
gc.collect()
artists = tracks.m4a_artist.fillna("").map(normalize).to_numpy()
popularity = (
    pd.to_numeric(tracks.spotify_popularity, errors="coerce").fillna(-1).to_numpy()
)
embedding_table = (
    pd.read_parquet(DATA_DIR / "track_embeddings.parquet")
    .set_index("m4a_id")
    .reindex(track_ids)
)
embeddings = np.stack(embedding_table.muq_embedding).astype(np.float32)
embeddings /= np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-8)
del embedding_table
metadata_documents = (
    tracks[[c for c in TEXT_COLUMNS if c not in ["lyrics", "query_document"]]]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .tolist()
)
query_documents = tracks.query_document.tolist()


Треков: 64,016; с train-запросами: 28,335


## 4. Готовые индексы

In [4]:
retriever = bm25s.BM25.load(str(BASE_DIR / "bm25_index"), mmap=True)
short_bm25 = bm25s.BM25.load(str(SEARCH_DIR / "metadata_bm25_index"), mmap=True)
assert np.array_equal(np.load(BASE_DIR / "track_ids.npy"), track_ids)
text_sources = {}
for name, folder, prefix in [
    ("tfidf", BASE_DIR, "tfidf"),
    ("query_tfidf", SEARCH_DIR, "query_tfidf"),
    ("metadata_char", SEARCH_DIR, "metadata_char"),
]:
    vectorizer = joblib.load(folder / f"{prefix}_vectorizer.joblib")
    matrix = load_npz(folder / f"{prefix}_tracks.npz").tocsr()
    text_sources[name] = (vectorizer, matrix.T.tocsr())


## 5. История пользователей

In [5]:
def top_indices(scores, k=MAX_K, positive=True):
    ids = np.flatnonzero(scores > 0) if positive else np.arange(len(scores))
    if len(ids) > k:
        threshold = np.partition(scores[ids], -k)[-k]
        larger = ids[scores[ids] > threshold]
        tied = ids[scores[ids] == threshold]
        ids = np.concatenate([larger, tied[: k - len(larger)]])
    return ids[np.lexsort((ids, -scores[ids]))].astype(np.int32)


def prepare_users(filename, user_ids, cutoff):
    table = pd.read_parquet(
        DATA_DIR / filename, columns=["user_id", "history", "history_ts"]
    )
    table = table[table.user_id.isin(user_ids)]
    cutoff = int(pd.Timestamp(cutoff, tz="UTC").timestamp())
    result = {}
    for row in tqdm(table.itertuples(index=False), total=len(table), desc=filename):
        counts = Counter()
        last = {}
        for track, timestamp in zip(row.history, row.history_ts):
            assert timestamp < cutoff
            idx = id_to_idx.get(track)
            if idx is not None:
                counts[idx] += 1
                last[idx] = int(timestamp)
        ids = np.array(list(counts), dtype=np.int32)
        freq = np.array([counts[i] for i in ids], dtype=np.float32)
        days = np.array([(cutoff - last[i]) / 86400 for i in ids])
        sources = {}
        for name, weights in [
            ("history", np.log1p(freq) * np.exp2(-days / 180)),
            ("history_recent", np.log1p(freq) * np.exp2(-days / 30)),
            ("history_frequency", freq),
        ]:
            sources[name] = ids[np.lexsort((ids, -weights))][:MAX_K]
        for name, weights in [
            ("audio_taste", freq),
            ("audio_recent", np.log1p(freq) * np.exp2(-days / 30)),
        ]:
            if not len(ids):
                sources[name] = np.array([], dtype=np.int32)
                continue
            centroid = np.einsum(
                "i,ij->j", weights.astype(np.float32), embeddings[ids], optimize=False
            )
            centroid /= max(float(np.linalg.norm(centroid)), 1e-8)
            scores = np.einsum("ij,j->i", embeddings, centroid, optimize=False)
            sources[name] = top_indices(scores)
        artist_weights = Counter()
        for idx, count in counts.items():
            if artists[idx]:
                artist_weights[artists[idx]] += count
        # Сначала более любимый артист, внутри артиста — Spotify popularity.
        scores = np.array([artist_weights.get(a, 0) for a in artists], dtype=np.float32)
        eligible = np.flatnonzero(scores > 0)
        sources["favorite_artists"] = eligible[
            np.lexsort((eligible, -popularity[eligible], -scores[eligible]))
        ][:MAX_K].astype(np.int32)
        sources["_counts"] = counts
        sources["_last"] = last
        sources["_cutoff"] = cutoff
        sources["_artist_counts"] = artist_weights
        result[row.user_id] = sources
    assert set(user_ids) <= set(result)
    return result


train_users = prepare_users(
    "train.parquet", set(rank_inputs.user_id) | set(valid_inputs.user_id), "2020-01-21"
)
test_users = prepare_users(
    "test_public.parquet", set(test_inputs.user_id), "2020-02-20"
)
for users in [train_users, test_users]:
    for user in users.values():
        counts = user["_counts"]
        indices = np.array(list(counts), dtype=np.int32)
        weights = np.array(list(counts.values()), dtype=np.float32)
        centroid = np.einsum("i,ij->j", weights, embeddings[indices], optimize=False)
        centroid /= max(float(np.linalg.norm(centroid)), 1e-8)
        user["_centroid"] = centroid
        user["_retrieval_scores"] = {
            i: float(
                np.log1p(c)
                * np.exp2(-(user["_cutoff"] - user["_last"][i]) / 86400 / 180)
            )
            for i, c in counts.items()
        }


## 6. Адаптивный отбор 150
Для известного query_type используем зафиксированный набор источников и веса; для неизвестного — fallback. Нулевые оценки не дополняем случайными треками. Цель не используется при отборе.

In [6]:
def load_sources(split, inputs):
    result = {}
    for name in SOURCE_NAMES:
        frame = pd.read_parquet(CACHE_DIR / f"{split}_{name}.parquet").set_index(
            "query_id"
        )
        result[name] = frame.loc[inputs.query_id, "track_indices"].tolist()
    return result


valid_sources = load_sources("validation", valid_inputs)
test_sources = load_sources("test", test_inputs)


def retrieval_batches(inputs, users, cached=None):
    for start in range(0, len(inputs), BATCH_SIZE):
        batch = inputs.iloc[start : start + BATCH_SIZE]
        queries = batch["query"].tolist()
        vectors = {
            name: vectorizer.transform(queries)
            for name, (vectorizer, _) in text_sources.items()
        }
        scores = {
            name: (vectors[name] @ transpose).toarray()
            for name, (_, transpose) in text_sources.items()
        }
        source_batches = {}
        if cached is None:
            tokens = bm25s.tokenize(queries, stopwords="en", show_progress=False)
            for name, index in [("bm25", retriever), ("metadata_bm25", short_bm25)]:
                ids, values = index.retrieve(
                    tokens, k=TOP_K, sorted=True, n_threads=THREADS, show_progress=False
                )
                source_batches[name] = [
                    x[y > 0].astype(np.int32) for x, y in zip(ids, values)
                ]
        for offset, row in enumerate(batch.itertuples(index=False)):
            if cached is not None:
                sources = {name: cached[name][start + offset] for name in SOURCE_NAMES}
            else:
                sources = {
                    name: source_batches[name][offset] for name in source_batches
                }
                sources.update(
                    {
                        name: top_indices(values[offset])
                        for name, values in scores.items()
                    }
                )
                sources.update(
                    {
                        name: users[row.user_id][name]
                        for name in SOURCE_NAMES
                        if name not in sources
                    }
                )
            yield row, sources, {
                name: values[offset] for name, values in scores.items()
            }


def union_and_rrf(sources):
    union = np.unique(np.concatenate([sources[n] for n in SOURCE_NAMES])).astype(
        np.int32
    )
    rrf = np.zeros(len(track_ids), dtype=np.float32)
    for ids in sources.values():
        rrf[ids] += 1 / (RRF_CONSTANT + np.arange(1, len(ids) + 1))
    return union, rrf


def selection_scores(query_type, sources):
    policy = CAP_CONFIG["by_query_type"].get(str(query_type), CAP_CONFIG["fallback"])
    scores = np.zeros(len(track_ids), dtype=np.float32)
    text_names = {"bm25", "tfidf", "metadata_bm25", "metadata_char", "query_tfidf"}
    for name in policy["generators"]:
        ids = sources[name]
        weight = policy["text_weight"] if name in text_names else 1
        scores[ids] += weight / (policy["rrf_constant"] + np.arange(1, len(ids) + 1))
    return scores


def cap_candidates(query_type, sources, union):
    scores = selection_scores(query_type, sources)
    eligible = union[scores[union] > 0]
    if len(eligible) > CANDIDATE_LIMIT:
        threshold = np.partition(scores[eligible], -CANDIDATE_LIMIT)[-CANDIDATE_LIMIT]
        above = eligible[scores[eligible] > threshold]
        tied = eligible[scores[eligible] == threshold]
        eligible = np.concatenate([above, tied[: CANDIDATE_LIMIT - len(above)]])
    ids = np.sort(eligible).astype(np.int32)
    assert len(ids) <= CANDIDATE_LIMIT and len(ids) == len(np.unique(ids))
    return ids


print("Candidate cap:", CANDIDATE_LIMIT)
display(pd.read_csv(POLICY_DIR / "validation_policy_by_type.csv"))


Candidate cap: 150


,query_type,n_queries,recall,generators,rrf_constant,text_weight
0,audio_attributes,339,0.477876,bm25+history+favorite_artists+history_frequenc...,60,2
1,complex,335,0.850746,bm25+tfidf+history+query_tfidf+favorite_artist...,60,2
2,era_region,313,0.785942,bm25+history_frequency+history_recent,60,2
3,exact,315,0.996825,metadata_char,20,1
4,genre,309,0.731392,bm25+history+query_tfidf+favorite_artists+hist...,60,2
5,lyrics_recall,259,0.980695,bm25+history,60,4
6,lyrics_theme,259,0.714286,bm25+history+history_frequency,60,2
7,mood,262,0.545802,bm25+history+history_recent,60,1
8,negative_constraint,319,0.514107,bm25+history+history_recent,60,1
9,novelty,185,0.621622,bm25+query_tfidf,60,1


## 7. Текстовые признаки без Python-прохода по документам
Бинарные разреженные матрицы дают точное пересечение слов. Для каждого запроса вычисляем coverage всего каталога и затем выбираем нужные строки. Остальные признаки совпадают по смыслу с предыдущим ранкером; новые признаки отражают все 11 источников.

In [7]:
def word_set(text):
    return set(normalize(text).split())


def bigram_set(text):
    words = normalize(text).split()
    return set(zip(words, words[1:]))


def trigram_set(text):
    text = normalize(text)
    return {text[i : i + 3] for i in range(max(len(text) - 2, 0))}


def dice(left, right):
    return 2 * len(left & right) / max(len(left) + len(right), 1)


word_vectorizer = CountVectorizer(
    analyzer=lambda x: list(word_set(x)), binary=True, dtype=np.float32
)
word_vectorizer.fit(documents)
coverage_documents = {
    "document_coverage": documents,
    "title_coverage": tracks.m4a_song.fillna("").tolist(),
    "artist_coverage": tracks.m4a_artist.fillna("").tolist(),
    "tag_coverage": tracks[["m4a_genres_full", "m4a_tags_full"]]
    .fillna("")
    .agg(" ".join, axis=1)
    .tolist(),
    "lyrics_coverage": tracks.lyrics.fillna("").tolist(),
}
coverage_matrices = {
    name: word_vectorizer.transform(texts).tocsr()
    for name, texts in coverage_documents.items()
}
word_lengths = {
    name: np.asarray(matrix.sum(axis=1)).ravel()
    for name, matrix in coverage_matrices.items()
}
normalized_titles = tracks.m4a_song.fillna("").map(normalize).to_numpy()
title_trigrams = [trigram_set(x) for x in normalized_titles]
artist_trigrams = [trigram_set(x) for x in artists]
lyrics_bigrams = [bigram_set(x) for x in tracks.lyrics.fillna("")]
del documents, metadata_documents, query_documents, index_rows, coverage_documents
popularity = popularity.astype(np.float32)
gc.collect()


def feature_frame(row, sources, text_scores, user, ids, union, all_rrf):
    query_text = normalize(row.query)
    query_words = word_set(row.query)
    query_bigram = bigram_set(row.query)
    query_trigram = trigram_set(row.query)
    tokens = bm25s.tokenize(
        [row.query], stopwords="en", return_ids=False, show_progress=False
    )[0]
    bm25 = (
        retriever.get_scores(tokens)
        if tokens
        else np.zeros(len(track_ids), dtype=np.float32)
    )
    counts = user["_counts"]
    total_count = max(sum(counts.values()), 1)
    last = user["_last"]
    history = user["_retrieval_scores"]
    ranks = {
        name: {int(idx): rank for rank, idx in enumerate(values, 1)}
        for name, values in sources.items()
    }
    b = bm25[ids]
    t = text_scores["tfidf"][ids]
    h = np.array([history.get(int(i), 0) for i in ids], dtype=np.float32)
    old_rrf = np.zeros(len(ids), dtype=np.float32)
    for name in ["bm25", "tfidf", "history"]:
        old_rrf += np.array(
            [
                1 / (RRF_CONSTANT + ranks[name][int(i)]) if int(i) in ranks[name] else 0
                for i in ids
            ],
            dtype=np.float32,
        )
    values = {
        "bm25_score": b,
        "bm25_rank": [ranks["bm25"].get(int(i), TOP_K + 1) for i in ids],
        "bm25_relative_score": b
        / max(float(bm25[sources["bm25"]].max()) if len(sources["bm25"]) else 0, 1e-8),
        "tfidf_score": t,
        "history_retrieval_score": h,
        "rrf_score": old_rrf,
        "spotify_popularity": popularity[ids],
        "history_track_log_count": [np.log1p(counts.get(int(i), 0)) for i in ids],
        "history_artist_share": [
            user["_artist_counts"].get(artists[i], 0) / total_count for i in ids
        ],
        "history_track_recency_days": [
            (user["_cutoff"] - last[int(i)]) / 86400 if int(i) in last else -1
            for i in ids
        ],
        "history_audio_cosine": np.einsum(
            "ij,j->i", embeddings[ids], user["_centroid"], optimize=False
        ),
        "tfidf_rank": [ranks["tfidf"].get(int(i), TOP_K + 1) for i in ids],
        "tfidf_relative_score": t / max(float(text_scores["tfidf"][union].max()), 1e-8),
        "history_rank": [ranks["history"].get(int(i), TOP_K + 1) for i in ids],
        "query_token_count": len(query_words),
        "title_phrase_match": [
            float(
                bool(normalized_titles[i])
                and f" {normalized_titles[i]} " in f" {query_text} "
            )
            for i in ids
        ],
        "artist_phrase_match": [
            float(bool(artists[i]) and f" {artists[i]} " in f" {query_text} ")
            for i in ids
        ],
        "lyrics_bigram_coverage": [
            len(query_bigram & lyrics_bigrams[i]) / max(len(query_bigram), 1)
            for i in ids
        ],
        "title_trigram_dice": [dice(query_trigram, title_trigrams[i]) for i in ids],
        "artist_trigram_dice": [dice(query_trigram, artist_trigrams[i]) for i in ids],
        "bm25_score_share": b / max(float(bm25[union].sum()), 1e-8),
        "tfidf_score_share": t / max(float(text_scores["tfidf"][union].sum()), 1e-8),
        "history_score_normalized": h / max(max(history.values(), default=0), 1e-8),
        "all_rrf_score": all_rrf[ids],
    }
    query_vector = word_vectorizer.transform([row.query])
    for name, matrix in coverage_matrices.items():
        overlap = (matrix[ids] @ query_vector.T).toarray().ravel()
        values[name] = overlap / max(len(query_words), 1)
        if name == "title_coverage":
            values["title_token_recall"] = overlap / np.maximum(
                word_lengths[name][ids], 1
            )
        if name == "artist_coverage":
            values["artist_token_recall"] = overlap / np.maximum(
                word_lengths[name][ids], 1
            )
        if name == "tag_coverage":
            values["tag_jaccard"] = overlap / np.maximum(
                len(query_words) + word_lengths[name][ids] - overlap, 1
            )
    source_count = np.zeros(len(ids), dtype=np.float32)
    for name in SOURCE_NAMES:
        rank = np.array(
            [ranks[name].get(int(i), TOP_K + 1) for i in ids], dtype=np.float32
        )
        hit = (rank <= TOP_K).astype(np.float32)
        values[f"{name}_source_rank"] = rank
        values[f"{name}_source_hit"] = hit
        source_count += hit
    values["all_source_count"] = source_count
    selection = selection_scores(row.query_type, sources)[ids]
    order = np.lexsort((ids, -selection))
    selection_ranks = np.empty(len(ids), dtype=np.float32)
    selection_ranks[order] = np.arange(1, len(ids) + 1)
    values["selection_rrf_score"] = selection
    values["selection_rank"] = selection_ranks
    frame = pd.DataFrame(values, columns=NUMERIC_FEATURES, dtype=np.float32)
    frame["query_type"] = str(row.query_type)
    assert np.isfinite(frame[NUMERIC_FEATURES].to_numpy()).all()
    return frame[FEATURES]


## 8. Обучающие группы
Каждая группа содержит всех кандидатов адаптивного отбора, до 150. Для Pool пропускаем группы без цели; в итоговой оценке такие запросы остаются в знаменателе. Негативы не семплируются.

In [8]:
def make_dataset(inputs, users, qrels, cached=None):
    targets = qrels.set_index("query_id").target_m4a_id.map(id_to_idx)
    parts = []
    labels = []
    groups = []
    found = 0
    group_index = 0
    iterator = retrieval_batches(inputs, users, cached)
    for row, sources, text_scores in tqdm(
        iterator, total=len(inputs), desc="150 candidate features"
    ):
        union, rrf = union_and_rrf(sources)
        ids = cap_candidates(row.query_type, sources, union)
        target = targets.loc[row.query_id]
        if target not in ids:
            continue
        frame = feature_frame(
            row, sources, text_scores, users[row.user_id], ids, union, rrf
        )
        parts.append(frame)
        labels.append((ids == target).astype(np.int8))
        groups.append(np.full(len(ids), group_index, dtype=np.int32))
        group_index += 1
        found += 1
    features = pd.concat(parts, ignore_index=True)
    label = np.concatenate(labels)
    group = np.concatenate(groups)
    assert label.sum() == found and np.all(np.diff(group) >= 0)
    print(
        "Positive groups:", found, "/", len(inputs), "rows:", len(features), flush=True
    )
    return features, label, group


train_x, train_y, train_groups = make_dataset(rank_inputs, train_users, train_qrels)
valid_x, valid_y, valid_groups = make_dataset(
    valid_inputs, train_users, train_qrels, valid_sources
)
training_counts = dict(
    train_queries=len(rank_inputs),
    train_positive_groups=int(train_y.sum()),
    train_rows=len(train_x),
    validation_queries=len(valid_inputs),
    validation_positive_groups=int(valid_y.sum()),
    validation_rows=len(valid_x),
)


Positive groups: 20688 / 30000 rows: 3102287


Positive groups: 2834 / 4000 rows: 425089


## 9. CatBoost
YetiRank, 350 итераций, depth=6, early stopping на validation-группах с найденной целью. Затем считаем отдельные метрики по всем запросам.

In [9]:
train_pool = Pool(
    train_x,
    label=train_y,
    group_id=train_groups,
    cat_features=CAT_FEATURES,
    feature_names=FEATURES,
)
valid_pool = Pool(
    valid_x,
    label=valid_y,
    group_id=valid_groups,
    cat_features=CAT_FEATURES,
    feature_names=FEATURES,
)
train_pool.quantize(border_count=128, random_seed=SEED)
train_pool.save_quantization_borders(str(OUT_DIR / "quantization_borders.tsv"))
valid_pool.quantize(input_borders=str(OUT_DIR / "quantization_borders.tsv"))
del train_x, valid_x, train_y, valid_y, train_groups, valid_groups
gc.collect()
model = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=20",
    iterations=ITERATIONS,
    depth=6,
    learning_rate=0.06,
    random_seed=SEED,
    thread_count=THREADS,
    l2_leaf_reg=5,
    one_hot_max_size=20,
    train_dir=str(OUT_DIR / "catboost_info"),
)
model.fit(
    train_pool,
    eval_set=valid_pool,
    early_stopping_rounds=50,
    use_best_model=True,
    verbose=50,
)
model.save_model(str(OUT_DIR / "catboost_ranker.cbm"))
(OUT_DIR / "feature_names.json").write_text(json.dumps(FEATURES, indent=2))
pd.DataFrame(
    {
        "feature": FEATURES,
        "importance": model.get_feature_importance(
            train_pool, type="PredictionValuesChange"
        ),
    }
).sort_values("importance", ascending=False).to_csv(
    OUT_DIR / "feature_importance.csv", index=False
)
print("Trees:", model.tree_count_, flush=True)
del train_pool, valid_pool
gc.collect()


0:	test: 0.3886915	best: 0.3886915 (0)	total: 800ms	remaining: 4m 39s


50:	test: 0.5377212	best: 0.5377212 (50)	total: 36.8s	remaining: 3m 35s


100:	test: 0.5493457	best: 0.5493990 (99)	total: 1m 13s	remaining: 3m


150:	test: 0.5527028	best: 0.5536993 (135)	total: 1m 50s	remaining: 2m 25s


200:	test: 0.5550616	best: 0.5550616 (200)	total: 2m 31s	remaining: 1m 52s


250:	test: 0.5557345	best: 0.5570251 (222)	total: 3m 14s	remaining: 1m 16s


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5570250613
bestIteration = 222

Shrink model to first 223 iterations.


Trees: 223


33

## 10. Оценка validation
Считаем recall, nDCG@20 и Hit@20 по всем 4 000 запросов. Для similar_to исключения применяются только при оценке. Дополнительно сохраняем recall большого исходного union.

In [10]:
def evaluate_full(inputs, users, qrels, cached, split):
    judgments = qrels.set_index("query_id")
    rows = []
    raw_predictions = []
    for row, sources, text_scores in tqdm(
        retrieval_batches(inputs, users, cached),
        total=len(inputs),
        desc=f"{split} 150 candidates",
    ):
        union, rrf = union_and_rrf(sources)
        ids = cap_candidates(row.query_type, sources, union)
        frame = (
            feature_frame(
                row, sources, text_scores, users[row.user_id], ids, union, rrf
            )
            if len(ids)
            else pd.DataFrame(columns=FEATURES)
        )
        scores = (
            model.predict(frame, thread_count=THREADS)
            if len(ids)
            else np.array([], dtype=float)
        )
        raw_order = np.lexsort((ids, -scores))[:20]
        raw_predictions.append(
            {
                "query_id": row.query_id,
                "top20": track_ids[ids[raw_order]].tolist(),
                "response": "",
            }
        )
        judgment = judgments.loc[row.query_id]
        keep = np.ones(len(ids), dtype=bool)
        if row.query_type == "similar_to":
            if pd.notna(judgment.exclude_m4a_id):
                keep &= track_ids[ids] != judgment.exclude_m4a_id
            if pd.notna(judgment.exclude_artist) and normalize(judgment.exclude_artist):
                keep &= artists[ids] != normalize(judgment.exclude_artist)
        eligible = ids[keep]
        eligible_scores = scores[keep]
        order = np.lexsort((eligible, -eligible_scores))
        target = id_to_idx[judgment.target_m4a_id]
        hits = np.flatnonzero(eligible[order[:20]] == target)
        rank = int(hits[0]) + 1 if len(hits) else None
        rows.append(
            {
                "query_id": row.query_id,
                "query_type": row.query_type,
                "candidate_count": len(ids),
                "retrieval_union_count": len(union),
                "recall_uncapped": float(target in union),
                "eligible_candidate_count": len(eligible),
                "recall_union": float(target in eligible),
                "catboost_ndcg_at_20": 1 / np.log2(rank + 1) if rank else 0.0,
                "catboost_hit_at_20": float(rank is not None),
            }
        )
    metrics = pd.DataFrame(rows)
    assert len(metrics) == len(inputs) and metrics.query_id.is_unique
    metrics.to_parquet(OUT_DIR / f"{split}_metrics_per_query.parquet", index=False)
    metrics.groupby("query_type").agg(
        n_queries=("query_id", "size"),
        recall_union=("recall_union", "mean"),
        catboost_ndcg_at_20=("catboost_ndcg_at_20", "mean"),
        catboost_hit_at_20=("catboost_hit_at_20", "mean"),
    ).to_csv(OUT_DIR / f"{split}_metrics_by_type.csv")
    pd.DataFrame(raw_predictions).to_parquet(
        OUT_DIR / f"{split}_raw_top20.parquet", index=False
    )
    overall = metrics.drop(columns=["query_id", "query_type"]).mean().to_dict()
    overall["n_queries"] = len(inputs)
    display(pd.DataFrame([overall]))
    return overall


validation_metrics = evaluate_full(
    valid_inputs, train_users, train_qrels, valid_sources, "validation"
)

assert np.isclose(validation_metrics["recall_union"], 0.7085)


,candidate_count,retrieval_union_count,recall_uncapped,eligible_candidate_count,recall_union,catboost_ndcg_at_20,catboost_hit_at_20,n_queries
0,149.99675,5446.2325,0.9515,148.36325,0.7085,0.419214,0.57275,4000


## 11. Public test
После сохранения модели читаем test_qrels и проверяем фиксированный ранкер на прежних 10 000 запросов. Сырые top20 сохраняются без фильтров из qrels и с пустым response; это не готовый сабмит.

In [11]:
test_qrels = pd.read_parquet(DATA_DIR / "test_public_qrels.parquet")
test_metrics = evaluate_full(test_inputs, test_users, test_qrels, test_sources, "test")
assert test_metrics["candidate_count"] <= CANDIDATE_LIMIT
assert np.isclose(test_metrics["recall_union"], 0.6833)
assert np.isclose(test_metrics["recall_uncapped"], 0.9489)
config = dict(
    seed=SEED,
    generators=SOURCE_NAMES,
    top_k_per_source=TOP_K,
    candidate_limit=CANDIDATE_LIMIT,
    cap_config=CAP_CONFIG,
    negative_sampling=False,
    iterations=ITERATIONS,
    features=FEATURES,
    trees=model.tree_count_,
    early_stopping_validation="all capped candidates",
    evaluation="all 150 candidates after cap",
    **training_counts
)
results = dict(config=config, validation=validation_metrics, test=test_metrics)
(OUT_DIR / "metrics.json").write_text(json.dumps(results, ensure_ascii=False, indent=2))
display(pd.read_csv(OUT_DIR / "feature_importance.csv").head(20))
print("Model and reports saved to:", OUT_DIR)


,candidate_count,retrieval_union_count,recall_uncapped,eligible_candidate_count,recall_union,catboost_ndcg_at_20,catboost_hit_at_20,n_queries
0,149.9753,5461.6247,0.9489,148.4566,0.6833,0.413624,0.5575,10000


,feature,importance
0,query_type,16.126208
1,bm25_relative_score,9.522626
2,history_score_normalized,8.429501
3,tag_jaccard,8.296817
4,selection_rank,7.762771
5,history_artist_share,5.413051
6,all_source_count,5.391243
7,history_retrieval_score,5.076605
8,tag_coverage,5.048960
9,lyrics_bigram_coverage,4.510759


Model and reports saved to: /Users/aleksei/Desktop/projects_vs_code/baseline_sota2026/adaptive150_catboost_artifacts


## 12. Проверки
Проверяем признаки сохранённой модели и 20 уникальных треков на запрос. Recall и nDCG относятся к выдаче из 150 кандидатов, а не исходному union из 5,5 тыс. треков.

In [12]:
loaded = CatBoostRanker()
loaded.load_model(str(OUT_DIR / "catboost_ranker.cbm"))
assert loaded.feature_names_ == FEATURES
predictions = pd.read_parquet(OUT_DIR / "test_raw_top20.parquet")
assert len(predictions) == TEST_LIMIT and predictions.query_id.is_unique
assert all(len(ids) == 20 and len(set(ids)) == 20 for ids in predictions.top20)
print(f"Validation nDCG@20: {validation_metrics['catboost_ndcg_at_20']:.6f}")
print(f"Public-test nDCG@20: {test_metrics['catboost_ndcg_at_20']:.6f}")
print(f"Public-test Hit@20: {test_metrics['catboost_hit_at_20']:.4%}")
print(f"Public-test candidate recall: {test_metrics['recall_union']:.4%}")


Validation nDCG@20: 0.419214
Public-test nDCG@20: 0.413624
Public-test Hit@20: 55.7500%
Public-test candidate recall: 68.3300%
